# 📊 Model Evaluation

A practical notebook for checking how well classification models perform.

## 🎯 What I am learning
- Accuracy
- Precision
- Recall
- F1 Score
- Confusion Matrix
- Classification Report
- Cross-Validation
- Comparing two models

### Workflow

**Dataset → Train/Test Split → Train Models → Evaluate → Cross-Validate → Compare Models**

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load the Dataset

For this notebook I am using the **Iris dataset** because it is small,
clean and useful for understanding model evaluation without adding
unnecessary preprocessing complexity.

In [ ]:
iris = load_iris(as_frame=True)

data = iris.frame

print("Dataset shape:", data.shape)
print("Classes:", list(iris.target_names))

display(data.head())

Dataset shape: (150, 5)
Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  target
0                5.1               3.5                1.4               0.2       0
1                4.9               3.0                1.4               0.2       0
2                4.7               3.2                1.3               0.2       0
3                4.6               3.1                1.5               0.2       0
4                5.0               3.6                1.4               0.2       0

## 2. Separate Features and Target

In [ ]:
X = data.drop("target", axis=1)
y = data["target"]

print("Features:", list(X.columns))
print("Target:", y.name)

Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target: target


## 3. Train/Test Split

I keep 20% of the data for final testing. The test data is not used while
training the models.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 120
Testing samples: 30


## 4. Models

I will compare two common classification models:

1. Logistic Regression
2. Random Forest Classifier

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

for name in models:
    print(name)

Logistic Regression
Random Forest


## 5. Evaluate Accuracy, Precision, Recall and F1

These metrics give different views of model performance.

- **Accuracy:** overall correct predictions
- **Precision:** correctness of positive/class predictions
- **Recall:** how many actual cases were identified
- **F1:** balance between precision and recall

In [ ]:
evaluation = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    evaluation.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted"),
        "Recall": recall_score(y_test, y_pred, average="weighted"),
        "F1 Score": f1_score(y_test, y_pred, average="weighted")
    })

evaluation_df = pd.DataFrame(evaluation)

evaluation_df.round(3)

              Model  Accuracy  Precision  Recall  F1 Score
Logistic Regression     0.967      0.970   0.967     0.967
      Random Forest     0.900      0.902   0.900     0.900

## 6. Compare the Models

The comparison makes it easier to choose the stronger model for this
dataset instead of looking at only one metric.

In [ ]:
best_model_name = evaluation_df.sort_values(
    "F1 Score",
    ascending=False
).iloc[0]["Model"]

print("Best model based on F1 Score:", best_model_name)

Best model based on F1 Score: Logistic Regression


## 7. Confusion Matrix

The confusion matrix shows how the selected model classified each class.

In [ ]:
best_model = models[best_model_name]
best_predictions = best_model.predict(X_test)

cm = confusion_matrix(y_test, best_predictions)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]


## 8. Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        best_predictions,
        target_names=iris.target_names
    )
)

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30


## 9. Cross-Validation

A single train/test split can sometimes give an incomplete picture.
Cross-validation trains and evaluates the model across multiple splits.

Here I use **5-fold cross-validation**.

In [ ]:
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=5
    )
    cv_results[name] = scores

cv_summary = pd.DataFrame({
    "Model": list(cv_results.keys()),
    "CV Mean Accuracy": [
        scores.mean() for scores in cv_results.values()
    ],
    "CV Std": [
        scores.std() for scores in cv_results.values()
    ]
})

cv_summary.round(3)

              Model  CV Mean Accuracy  CV Std
Logistic Regression             0.973   0.025
      Random Forest             0.967   0.021

## 10. Final Evaluation

### What I learned

Model evaluation is not just about getting the highest accuracy.
Different metrics can tell us different things about model behaviour.

For classification, I should check:

**Accuracy + Precision + Recall + F1 + Confusion Matrix + Cross-Validation**

### 🔄 Final Workflow

**Train → Predict → Evaluate → Cross-Validate → Compare → Select Model**

## 🚀 Next Step

**Model Tuning** — improving the selected model using hyperparameters
and systematic search.